# **Import libraries**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    ParameterSampler
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error

from xgboost import XGBRegressor
from scipy.stats import t as tdist

# **Import datasets**

In [ ]:
# Import the merged csv file of image analysis parameters + bioreactor process parameters

csv_file_path = (".../Merged_df.csv")

df_ML = pd.read_csv(csv_file_path)

# one-hot incoding
# 1. Inoculum: 'Preculture' or 'Conidia' → One-hot
df_ML['Inoculum'] = df_ML['Inoculum'].map({'Conidia': 0, 'Pre-culture': 1})

# 2. Stirring rate: extract numeric from 'e.g. "200 rpm"'
df_ML['Stirring rate'] = df_ML['Stirring rate'].str.extract(r'(\d+)').astype(float)

# 3. Overpressure: 'yes' = 1, 'no' = 0, '-'
df_ML['Overpressure'] = df_ML['Overpressure'].astype(str).str.lower().map({'yes': 1, 'no': 0, '-': 0})

# 4. Medium: e.g., 'standard', others → One-hot encode
df_ML['Medium_Nitrogen'] = df_ML['Medium'].str.contains("Nitrogen", case=False).astype(int)
df_ML['Medium_Phosphate'] = df_ML['Medium'].str.contains("Phosphate", case=False).astype(int)

df_ML.rename(columns={
    'Post-induction time [h]': 'Post-induction time',
    'Viscosity [Pa s]': 'Viscosity',
    'Glucose [g/l]': 'Glucose',
    'Integral OTR [mmol/l]': 'Integral OTR',
    'CDW [g/l]': 'CDW',
    r'Total area [$\mu m^2$]': 'Total area',
    r'Total perimeter [$\mu m$]': 'Total perimeter'
}, inplace=True)

df_ML

,Experiment,Bioreactor,Imaging time [h],Wellplate,Count per image,Total area [$\mu m^2$],Total perimeter [$\mu m$],Degree centrality,Inoculum,Stirring rate,...,CDW [g/l],PY/CDW [mg/g],qP = PY/CDW [mg/g/h],STY [mg/l/h],Glucose [g/l],Induction time [h],Integral OTR [mmol/l],ESF₁₀ disk diameter [µm],Medium_Nitrogen,Medium_Phosphate
0,AP00425,F3 0125 AN,144.0,XY05,336,1515049.76,321743.78,0.70,1,550.0,...,20.5,4.30,0.03,0.61,4.7,10.0,1385.92,90.5856,0,0
1,AP00425,F3 0125 AN,72.0,XY07,190,5206077.26,301554.09,0.92,1,550.0,...,17.4,0.90,0.01,0.21,46.5,10.0,574.98,603.9040,0,0
2,AP00425,F3 0125 AN,72.0,XY08,288,1563672.35,280141.64,0.74,1,550.0,...,17.4,0.90,0.01,0.21,46.5,10.0,574.98,75.4880,0,0
3,AP00425,F3 0125 AN,72.0,XY09,233,3167305.53,259614.71,0.88,1,550.0,...,17.4,0.90,0.01,0.21,46.5,10.0,574.98,362.3424,0,0
4,AP00425,F3 0125 AN,72.0,XY10,229,4083669.38,267359.58,0.91,1,550.0,...,17.4,0.90,0.01,0.21,46.5,10.0,574.98,407.6352,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,AP01224,K8 0224 AN,108.0,XY11,1050,978919.28,446086.14,0.35,0,1140.0,...,12.8,22.37,-0.95,2.65,0.0,19.0,NaN,7.5488,1,1
534,AP01224,K8 0224 AN,84.0,XY20,806,4166397.56,1741954.19,0.40,0,1140.0,...,10.8,49.11,-0.53,6.31,0.0,19.0,NaN,7.5488,1,1
535,AP01224,K8 0224 AN,84.0,XY19,807,3954341.59,1699443.33,0.38,0,1140.0,...,10.8,49.11,-0.53,6.31,0.0,19.0,NaN,7.5488,1,1
536,AP01224,K8 0224 AN,108.0,XY19,1301,1386405.80,633591.91,0.34,0,1140.0,...,12.8,22.37,-0.95,2.65,0.0,19.0,NaN,7.5488,1,1


# **XGBoost - leave-one-bioreactor-out cross-validation (LOBOCV)**

In [ ]:
target="Psilocybin [mg/l] Mean"
outdir="xgb_ablation_LOBOCV_inner20_bt" #Specify the output directory to save best models

# Ablation analysis
time=["Post-induction time"]
bio=["Inoculum","Stirring rate","Overpressure","Medium_Nitrogen","Medium_Phosphate"]
morph=["Count per image","Degree centrality","ESF₁₀ disk size"]

feature_sets={
    "Time":time,
    "Bioprocess":bio,
    "Morphology":morph,
    "Time+Bioprocess":time+bio,
    "Time+Morphology":time+morph,
    "Bioprocess+Morphology":bio+morph,
    "Combined model":time+bio+morph
}

space={
    "learning_rate":[.01,.03,.05,.1,.2,.3],
    "max_depth":[2,3,4,5,6],
    "min_child_weight":[1,3,5],
    "subsample":[.7,.8,.9,1],
    "colsample_bytree":[.7,.8,.9,1],
    "gamma":[0,.05,.1,.2],
    "reg_alpha":[0,.01,.1,.5],
    "reg_lambda":[.5,1,2,5]
}

params=list(ParameterSampler(space,n_iter=20,random_state=42))
os.makedirs(outdir,exist_ok=True)

def rtgroup(d):
    return d.Bioreactor.astype(str)+"_"+d["Post-induction time"].astype(str)

def weights(d):
    g=rtgroup(d); c=g.value_counts()
    return g.map(lambda x:1/c[x]).values

def agg(d,p):
    x=d[["Bioreactor","Post-induction time",target]].copy()
    x["Predicted"]=p
    x=x.groupby(["Bioreactor","Post-induction time"],as_index=False).agg(
        Measured=(target,"first"),
        Predicted=("Predicted","mean")
    )
    return r2_score(x.Measured,x.Predicted),mean_absolute_error(x.Measured,x.Predicted)

def save_block(d,p,held_out,reactor_col):
    z=d[["Bioreactor","Post-induction time",target]].copy()

    z.insert(0,"Held-out reactor",held_out)
    z["Predicted"]=p

    return z.rename(columns={
        "Bioreactor":reactor_col,
        "Post-induction time":"Post-induction time [h]",
        target:"Actual psilocybin [mg/l]",
        "Predicted":"Predicted psilocybin [mg/l]"
    })

for set_name,features in feature_sets.items():

    print(f"\n{'='*70}\n{set_name}\n{'='*70}")

    df=df_ML.dropna(subset=features+[target,"Bioreactor"]).copy()

    if df.Bioreactor.nunique()!=8:
        raise ValueError(f"Expected 8 reactors, found {df.Bioreactor.nunique()}")

    folder=os.path.join(outdir,set_name)
    os.makedirs(folder,exist_ok=True)

    train_folds=[]
    val_folds=[]
    test_folds=[]

    # OUTER = 8-fold leave-one-bioreactor-out
    for fold,(tr,te) in enumerate(
        GroupKFold(8).split(df,groups=df.Bioreactor),1
    ):

        outer_train=df.iloc[tr].copy()
        test=df.iloc[te].copy()

        train_r=sorted(outer_train.Bioreactor.unique())
        test_r=sorted(test.Bioreactor.unique())

        # The one held-out reactor identifies this fold/model
        held_out=test_r[0]

        print(f"\nFold {fold}/8")
        print("Held-out reactor:",held_out)
        print("Outer training reactors:",train_r)

        # INNER = 80/20 reactor-time split within outer training set
        itr,iva=next(
            GroupShuffleSplit(
                n_splits=1,
                test_size=.2,
                random_state=42
            ).split(
                outer_train,
                groups=rtgroup(outer_train)
            )
        )

        train=outer_train.iloc[itr].copy()
        val=outer_train.iloc[iva].copy()

        print(
            f"Inner train BT groups: {rtgroup(train).nunique()} | "
            f"Inner val BT groups: {rtgroup(val).nunique()}"
        )

        best=(np.inf,None,None,None,None)

        for i,par in enumerate(params,1):

            sc=StandardScaler()
            Xtr=sc.fit_transform(train[features])
            Xva=sc.transform(val[features])

            m=XGBRegressor(
                objective="reg:squarederror",
                n_estimators=300,
                early_stopping_rounds=20,
                eval_metric="mae",
                random_state=42,
                n_jobs=1,
                **par
            )

            m.fit(
                Xtr,
                train[target],
                sample_weight=weights(train),
                eval_set=[(Xva,val[target])],
                verbose=False
            )

            pva=np.maximum(m.predict(Xva),0)
            _,mae=agg(val,pva)
            nt=m.best_iteration+1

            if mae<best[0]:
                best=(mae,m,sc,par.copy(),nt)

            print(
                f"Param {i:02d}/{len(params)} | "
                f"Val MAE={mae:.3f} | Best={best[0]:.3f}"
            )

        inner_mae,m,sc,par,nt=best

        # Predictions from the selected best inner model
        ptr=np.maximum(m.predict(sc.transform(train[features])),0)
        pva=np.maximum(m.predict(sc.transform(val[features])),0)
        pte=np.maximum(m.predict(sc.transform(test[features])),0)

        tr_r2,tr_mae=agg(train,ptr)
        va_r2,va_mae=agg(val,pva)
        te_r2,te_mae=agg(test,pte)

        bundle={
            "model":m,
            "scaler":sc,
            "features":features,
            "feature_set":set_name,
            "fold":fold,
            "training_reactors":train_r,
            "test_reactors":test_r,
            "held_out_reactor":held_out,
            "inner_train_groups":sorted(rtgroup(train).unique()),
            "inner_validation_groups":sorted(rtgroup(val).unique()),
            "params":par,
            "n_estimators":nt,
            "inner_mae":inner_mae
        }

        joblib.dump(
            bundle,
            os.path.join(folder,f"fold_{fold}_held_out_{held_out}.joblib")
        )

        # Save raw predictions
        train_folds.append(
            save_block(train,ptr,held_out,"Training reactor")
        )

        val_folds.append(
            save_block(val,pva,held_out,"Validation reactor")
        )

        test_folds.append(
            save_block(test,pte,held_out,"Test reactor")
        )

        print(
            f"Train Agg MAE={tr_mae:.3f} | Train Agg R²={tr_r2:.3f} | "
            f"Val Agg MAE={va_mae:.3f} | Val Agg R²={va_r2:.3f} | "
            f"Test Agg MAE={te_mae:.3f} | Test Agg R²={te_r2:.3f}"
        )

    # Concatenate all 8 folds within this feature set
    train_pred=pd.concat(train_folds,ignore_index=True)
    val_pred=pd.concat(val_folds,ignore_index=True)
    test_pred=pd.concat(test_folds,ignore_index=True)

    train_pred.to_csv(
        os.path.join(folder,"train_predictions.csv"),
        index=False
    )

    val_pred.to_csv(
        os.path.join(folder,"validation_predictions.csv"),
        index=False
    )

    test_pred.to_csv(
        os.path.join(folder,"test_predictions.csv"),
        index=False
    )

    # Overall aggregated metrics for this feature set
    for name,pred,reactor_col in [
        ("TRAIN",train_pred,"Training reactor"),
        ("VALIDATION",val_pred,"Validation reactor"),
        ("TEST",test_pred,"Test reactor")
    ]:

        pm=pred.groupby(
            [reactor_col,"Post-induction time [h]"],
            as_index=False
        ).agg(
            Actual=("Actual psilocybin [mg/l]","first"),
            Predicted=("Predicted psilocybin [mg/l]","mean")
        )

        r2=r2_score(pm.Actual,pm.Predicted)
        mae=mean_absolute_error(pm.Actual,pm.Predicted)

        print(
            f"{set_name} {name} OVERALL | "
            f"Agg R²={r2:.3f} | Agg MAE={mae:.3f}"
        )

print("\nDONE")

# **Visualize train-validation-test split**

In [ ]:
# Visualize train-validation-test split TR: Train VL: Validation TS: Test

def rtgroup(d):
    return (
        d["Bioreactor"].astype(str)+"_"+
        d["Post-induction time"].astype(str)
    )

# Same data used for the 8-reactor LOBO
dev=df.copy()

reactors=sorted(dev["Bioreactor"].unique())
times=sorted(dev["Post-induction time"].unique())

fig,axs=plt.subplots(4,2,figsize=(18,20),sharex=True,sharey=True)
axs=axs.ravel()

cmap=ListedColormap(["lightgray","orange","steelblue"])

# OUTER: one entire bioreactor held out
for fold,(tr,te) in enumerate(
    GroupKFold(8).split(dev,groups=dev["Bioreactor"])
):

    outer_train=dev.iloc[tr].copy()
    outer_test=dev.iloc[te].copy()

    # INNER: 80/20 split by reactor-time groups
    itr,iva=next(
        GroupShuffleSplit(
            n_splits=1,
            test_size=.20,
            random_state=42
        ).split(
            outer_train,
            groups=rtgroup(outer_train)
        )
    )

    inner_train=outer_train.iloc[itr]
    inner_val=outer_train.iloc[iva]

    TR=set(rtgroup(inner_train))
    IV=set(rtgroup(inner_val))
    TEST=set(rtgroup(outer_test))

    A=np.full((len(reactors),len(times)),np.nan)

    for i,r in enumerate(reactors):
        for j,t in enumerate(times):
            g=f"{r}_{t}"

            if g in TR:
                A[i,j]=0
            if g in IV:
                A[i,j]=1
            if g in TEST:
                A[i,j]=2

    ax=axs[fold]
    ax.imshow(A,aspect="auto",vmin=0,vmax=2,cmap=cmap)

    n_all=rtgroup(outer_train).nunique()
    n_iv=rtgroup(inner_val).nunique()

    test_name=", ".join(map(str,outer_test.Bioreactor.unique()))

    ax.set_title(
        f"Fold {fold+1} | Test: {test_name}\n"
        f"Inner validation = {n_iv}/{n_all} reactor-time groups "
        f"({100*n_iv/n_all:.1f}%)"
    )

    ax.set_xticks(range(len(times)))
    ax.set_xticklabels(times,rotation=45,ha="right")

    ax.set_yticks(range(len(reactors)))
    ax.set_yticklabels(reactors)

    for i in range(len(reactors)):
        for j in range(len(times)):
            if not np.isnan(A[i,j]):
                ax.text(
                    j,i,
                    ["TR","VL","TS"][int(A[i,j])],
                    ha="center",
                    va="center",
                    fontsize=8
                )

fig.legend(
    handles=[
        Patch(facecolor="lightgray",label="Inner train"),
        Patch(facecolor="orange",label="Inner validation"),
        Patch(facecolor="steelblue",label="Outer test")
    ],
    loc="lower center",
    ncol=3,
    frameon=False
)

plt.tight_layout(rect=[0,.04,1,1])
plt.show()

# **Visualize LOBOCV results per bioreactor**

In [ ]:
plt.rcParams.update({
    "font.size":16,"axes.titlesize":18,"axes.labelsize":17,
    "xtick.labelsize":15,"ytick.labelsize":15,
    "legend.fontsize":14,"legend.title_fontsize":15
})

root="xgb_ablation_reactor_8fold_inner20_bt"

tcol="Post-induction time [h]"
ycol="Actual psilocybin [mg/l]"
pcol="Predicted psilocybin [mg/l]"

morph=["Count per image","Degree centrality","ESF₁₀ disk size"]
remove=["F6 1523 AN","F7 0823 AN"]

order=["Time","Bioprocess","Morphology","Time+Bioprocess",
       "Time+Morphology","Bioprocess+Morphology","Combined model"]

reactor_order=["F3 0125 AN","F8 0125 AN","K7 0124 AN","K8 0124 AN",
               "F9 0324 AN","F8 0424 AN","K7 0224 AN","K8 0224 AN"]

bioreactor_map={
    "F3 0125 AN":"STR 7 L – 550 rpm",
    "F8 0125 AN":"STR 7 L – 750 rpm",
    "K7 0124 AN":"P-STR 7.5 L – 850 rpm",
    "K8 0124 AN":"P-STR 7.5 L – 850 rpm (4 × NH₄⁺)",
    "F9 0324 AN":"STR 7 L – 550 rpm",
    "F8 0424 AN":"STR 7 L – 1006 rpm",
    "K7 0224 AN":"P-STR 7.5 L – 1140 rpm",
    "K8 0224 AN":"P-STR 7.5 L – 1140 rpm (4 × NH₄⁺, 3 × PO₄³⁻)"
}

# TEST predictions for all feature sets
preds={
    fs:pd.read_csv(os.path.join(root,fs,"ALL_TEST_PREDICTIONS.csv"))
    for fs in order
}

d=(df_ML.dropna(subset=morph+["Bioreactor"])
   .loc[lambda x:~x.Bioreactor.isin(remove)].copy())

# PCA
pca=PCA(2)
d[["PC1","PC2"]]=pca.fit_transform(
    StandardScaler().fit_transform(d[morph])
)

xr,yr=np.ptp(d.PC1),np.ptp(d.PC2)
xlim=(d.PC1.min()-.08*xr,d.PC1.max()+.08*xr)
ylim=(d.PC2.min()-.08*yr,d.PC2.max()+.08*yr)

reactors=sorted(d.Bioreactor.unique())
markers=["o","s","^","v","D","P","X","<",">","*","h","p"]
mk={r:markers[i%len(markers)] for i,r in enumerate(reactors)}

vals=np.concatenate([
    np.r_[preds[fs][ycol].dropna(),preds[fs][pcol].dropna()]
    for fs in order
])

pad=.05*np.ptp(vals)
right_ylim=(vals.min()-pad,vals.max()+pad)


def test_metrics(fs,rhold):

    q=preds[fs]
    q=q[q["Held-out reactor"]==rhold]

    q=(q.groupby(["Test reactor",tcol],as_index=False)
       .agg(Actual=(ycol,"first"),
            Predicted=(pcol,"mean")))

    r2=(r2_score(q.Actual,q.Predicted)
        if len(q)>1 and q.Actual.nunique()>1 else np.nan)

    mae=(mean_absolute_error(q.Actual,q.Predicted)
         if len(q) else np.nan)

    return r2,mae


def make_figure(selected,title):

    fig,ax=plt.subplots(
        4,3,figsize=(25,23.2),
        gridspec_kw={"width_ratios":[1.2,1.9,2.35],
                     "wspace":.38,"hspace":.52}
    )

    for i,rhold in enumerate(selected):

        # =========================================================
        # PCA
        # =========================================================
        for r in reactors:
            z=d[d.Bioreactor==r]
            h=r==rhold

            ax[i,0].scatter(
                z.PC1,z.PC2,
                marker=mk[r],
                s=85 if h else 34,
                alpha=.9 if h else .22,
                color="tab:blue" if h else "gray"
            )

        ax[i,0].set(
            xlim=xlim,ylim=ylim,
            xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
            ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
            title=f"{rhold}\n{bioreactor_map[rhold]}"
        )

        ax[i,0].grid(alpha=.15)

        # =========================================================
        # PERFORMANCE
        # =========================================================
        a=ax[i,1]

        r2=[]
        mae=[]

        for fs in order:
            rr,mm=test_metrics(fs,rhold)
            r2.append(rr)
            mae.append(mm)

        r2=np.array(r2,float)
        mae=np.array(mae,float)

        yy=np.arange(len(order))

        rp=np.clip(r2,0,1)
        mp=mae/max(np.nanmax(mae),1e-9)

        a.barh(
            yy,-rp,height=.58,
            color="tab:blue",
            edgecolor="black",
            linewidth=1,
            alpha=.7
        )

        a.barh(
            yy,mp,height=.58,
            color="tab:orange",
            edgecolor="black",
            linewidth=1,
            alpha=.7
        )

        a.axvline(0,color="black",lw=1)

        for k,(rv,mv) in enumerate(zip(r2,mae)):

            if pd.notna(rv):
                a.text(
                    -.04 if rv<0 else -rp[k]-.04,
                    k,
                    "<0" if rv<0 else f"{rv:.2f}",
                    ha="right",va="center",
                    fontsize=15,
                    color="tab:blue",
                    fontweight="bold"
                )

            if pd.notna(mv):
                a.text(
                    mp[k]+.04,k,f"{mv:.1f}",
                    ha="left",va="center",
                    fontsize=15,
                    color="tab:orange",
                    fontweight="bold"
                )

        a.set(
            xlim=(-1.35,1.35),
            ylim=(-.5,len(order)-.5),
            yticks=yy,
            yticklabels=order
        )

        a.invert_yaxis()
        a.set_xticks([])
        a.spines[["top","right","bottom"]].set_visible(False)

        a.text(
            .23,-.07,"R²",
            transform=a.transAxes,
            ha="center",
            fontweight="bold",
            fontsize=16,
            color="tab:blue"
        )

        a.text(
            .77,-.07,"MAE",
            transform=a.transAxes,
            ha="center",
            fontweight="bold",
            fontsize=16,
            color="tab:orange"
        )

        # =========================================================
        # WINNER
        # =========================================================
        perf=pd.DataFrame({
            "Feature_Set":order,
            "R2":r2,
            "MAE":mae
        }).dropna()

        nonneg=perf[perf.R2>=0]

        if len(nonneg)==0:
            winner_row=perf.sort_values("MAE").iloc[0]
        else:
            best_r2=nonneg.R2.max()
            close=nonneg[nonneg.R2>=best_r2-.02]
            winner_row=close.sort_values("MAE").iloc[0]

        winner=winner_row.Feature_Set
        winner_r2=winner_row.R2
        winner_mae=winner_row.MAE

        # =========================================================
        # TRUE
        # =========================================================
        qtrue=preds["Combined model"]
        qtrue=qtrue[qtrue["Held-out reactor"]==rhold]

        tt=(qtrue.groupby(tcol,as_index=False)
            .agg(True_Psilo=(ycol,"first"))
            .sort_values(tcol))

        ax[i,2].plot(
            tt[tcol],
            tt.True_Psilo,
            "-o",
            color="black",
            lw=3.2,
            ms=7,
            zorder=20
        )

        # =========================================================
        # OTHER FEATURE SETS
        # =========================================================
        for fs in order:

            if fs==winner:
                continue

            q=preds[fs]
            q=q[q["Held-out reactor"]==rhold]

            c=(q.groupby(tcol,as_index=False)
               .agg(Pred=(pcol,"mean"))
               .sort_values(tcol))

            ax[i,2].plot(
                c[tcol],c.Pred,
                "-o",
                color="gray",
                lw=1.2,
                ms=3.5,
                alpha=.28,
                zorder=1
            )

        # =========================================================
        # WINNER — INDIVIDUAL + MEAN + 95% CI
        # =========================================================
        q=preds[winner]
        q=q[q["Held-out reactor"]==rhold].copy()

        ax[i,2].scatter(
            q[tcol],
            q[pcol],
            color="tab:red",
            s=28,
            alpha=.22,
            zorder=3
        )

        c=(q.groupby(tcol)[pcol]
           .agg(["mean","std","count"])
           .reset_index()
           .sort_values(tcol))

        c["sem"]=c["std"]/np.sqrt(c["count"])

        c["ci"]=[
            tdist.ppf(.975,n-1)*s
            if n>1 and pd.notna(s)
            else 0
            for s,n in zip(c["sem"],c["count"])
        ]

        times=c[tcol].to_numpy()
        mean=c["mean"].to_numpy()
        ci=c["ci"].to_numpy()

        ax[i,2].fill_between(
            times,
            mean-ci,
            mean+ci,
            color="tab:red",
            alpha=.14,
            zorder=2
        )

        ax[i,2].plot(
            times,
            mean,
            "-o",
            color="tab:red",
            lw=3,
            ms=7,
            zorder=10
        )

        r2_text="R²<0" if winner_r2<0 else f"R²={winner_r2:.2f}"

        ax[i,2].set_title(
            f"Winner: {winner}\n"
            f"{r2_text} | MAE={winner_mae:.1f}",
            fontweight="bold",
            fontsize=17
        )

        ax[i,2].set(
            xlabel="Post-induction time [h]",
            ylabel="Psilocybin [mg/L]",
            ylim=right_ylim
        )

        ax[i,2].grid(alpha=.18)
        ax[i,2].spines[["top","right"]].set_visible(False)

        ax[i,2].legend(
            handles=[
                Line2D([0],[0],color="black",lw=3,
                       marker="o",label="True"),

                Line2D([0],[0],color="tab:red",
                       marker="o",linestyle="None",
                       alpha=.3,label="Individual predictions"),

                Line2D([0],[0],color="tab:red",lw=3,
                       marker="o",label="Mean prediction"),

                Patch(facecolor="tab:red",alpha=.14,
                      edgecolor="none",label="95% CI"),

                Line2D([0],[0],color="gray",
                       lw=1.3,alpha=.5,
                       label="Other feature sets")
            ],
            frameon=False,
            fontsize=13,
            loc="best"
        )

    fig.suptitle(
        title,
        fontsize=22,
        fontweight="bold",
        y=1.04
    )

    plt.subplots_adjust(
        left=.07,right=.98,
        top=.95,bottom=.04,
        wspace=.38,hspace=.52
    )

    for i in range(4):
        pos=ax[i,1].get_position()
        ax[i,1].set_position([
            pos.x0+.04,
            pos.y0,
            pos.width,
            pos.height
        ])

    fig.legend(
        handles=[
            Line2D(
                [0],[0],
                marker=mk[r],
                linestyle="None",
                markerfacecolor="gray",
                markeredgecolor="gray",
                markersize=12,
                label=r
            )
            for r in reactors
        ],
        loc="upper center",
        bbox_to_anchor=(.5,1.02),
        ncol=4,
        fontsize=16,
        title="Bioreactor / marker shape",
        title_fontsize=16,
        frameon=True
    )

    plt.show()


make_figure(
    reactor_order[:4],
    "Reactor-out CV — Pre-culture"
)

make_figure(
    reactor_order[4:],
    "Reactor-out CV — Conidia"
)